# Basic Financial Retrieval System

In [5]:
from pprint import pprint
import os
import getpass

In [6]:
# Load the PDF document
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("..\\data\\raw\\_10-K-2025-As-Filed.pdf")
reader = loader.load()


print(len(reader))

80


In [7]:
# split the document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(reader)

pprint([doc.page_content[:100] for doc in docs[:3]])

['UNITED STATES\n'
 'SECURITIES AND EXCHANGE COMMISSION\n'
 'Washington, D.C. 20549\n'
 'FORM 10-K\n'
 '(Mark One)\n'
 '☒    AN',
 'Title of each class\n'
 'Trading \n'
 'symbol(s) Name of each exchange on which registered\n'
 'Common Stock, $0.00',
 'Indicate by check mark whether the Registrant (1)\xa0has filed all reports '
 'required to be filed by Sect']


In [8]:
# API key
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

In [17]:
# Embedded the chunks and store them in a vector database
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="apple_10k",
    persist_directory="../chroma_db/apple_10k"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
# Retrieving from Vector DB
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

question = "What are the company's main risk factors?"
results = retriever.invoke(question)

print(results[0].page_content)



Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of 
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these 
risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially 
adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred 
previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement 
of all potential risks or uncertainties that the Company faces or may face in the future.
This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition 
and Results of Operations” and the consolidated financial statements and accompanying notes in Part II, Item 8, “Financial


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


context = "\n\n".join(
    [
        f"Source {i} | Page {doc.metadata.get('page')}:\n{doc.page_content}"
        for i, doc in enumerate(results, start=1)
    ]
)

prompt = f"""
You are a financial document analysis assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, say: "I could not find this in the retrieved sections."

Question:
{question}

Context:
{context}

Answer with:
1. A clear business-focused answer
2. Bullet points when helpful
3. Source citations using Source number and page number
"""

response = llm.invoke([HumanMessage(content=prompt)])

pprint(response.content)

("The company's main risk factors include:\n"
 '\n'
 '*   **Operational Disruptions:**\n'
 "    *   Interruption of the Company's and its suppliers' operations, retail "
 'stores, and facilities due to events such as fire, power shortages, nuclear '
 'power plant accidents, other industrial accidents, terrorist attacks, '
 'hostile acts, ransomware and other cybersecurity attacks, labor disputes, '
 'and public health issues (including pandemics like COVID-19). These events '
 'can make it difficult to manufacture and deliver products, create supply '
 'chain delays and inefficiencies, cause service slowdowns and outages, '
 'increase costs, and negatively impact consumer spending and demand. (Source '
 '2, Page 8; Source 3, Page 8)\n'
 '    *   Industrial accidents at suppliers and contract manufacturers, which '
 'could result in serious injuries or loss of life, business disruption, and '
 'harm to the Company’s reputation. (Source 3, Page 8)\n'
 '*   **Strategic Investments and Acq